# models2.ipynb — ISOT Dataset

Same pipeline as models1_full, adapted for the ISOT fake news dataset (Reuters real articles vs scraped fake news sites). Key differences from models1:

- **ISOT dataset**: 44k real news articles with actual semantic content. Real named entities, real claims, real search queries.
- **`ddgs` import**: `duckduckgo_search` was renamed upstream — updated to avoid the RuntimeWarning on every search call.
- **Cache dir created inside the function**: previously `os.makedirs` was in a separate cell and silently vanished on kernel restart, causing every cache write to fail with `FileNotFoundError`.
- **Entity-centric claim extraction**: replaces the SVO approach. Queries are built from named entity combinations (e.g. `'Manchin Biden Build Back Better'`) rather than dependency-parsed predicates (e.g. `'Manchin block plan'`). Search engines index by entity, not predicate.
- **Larger sample**: 500 articles (up from 200) — ISOT has 44k so we're no longer limited by dataset size.

**Setup**: Download ISOT from https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset  
Place `Fake.csv` and `True.csv` in a `data/` subfolder next to this notebook.

In [1]:
# pip install spacy ddgs sentence-transformers torch-geometric scikit-learn networkx
# python -m spacy download en_core_web_sm

import os, json, hashlib, math, time, sqlite3
from contextlib import contextmanager
from urllib.parse import urlparse
from datetime import datetime

import nltk
try:
    nltk.download('punkt_tab', quiet=True)
except Exception:
    nltk.download('punkt', quiet=True)

import spacy
nlp = spacy.load('en_core_web_sm')

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN
import numpy as np
import networkx as nx
from nltk.tokenize import sent_tokenize

ENCODER    = SentenceTransformer('all-MiniLM-L6-v2')
NLI_MODEL  = CrossEncoder('cross-encoder/nli-deberta-v3-small')

NLI_CONTRADICTION = 0
NLI_ENTAILMENT    = 1
NLI_NEUTRAL       = 2
EDGE_NEUTRAL       = 0
EDGE_ENTAILMENT    = 1
EDGE_CONTRADICTION = 2

FEATURE_DIM = 14

print('Setup complete.')

Setup complete.


## Step 1: Load ISOT Dataset

ISOT uses two separate CSVs — `True.csv` (Reuters articles) and `Fake.csv` (scraped fake news). We concat them, shuffle, and create a binary label. The dataset is well-balanced (~21k real, ~23k fake) and contains full article bodies with real named entities, dates, and verifiable claims — exactly what the pipeline needs.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

fake = pd.read_csv('data/Fake.csv')
real = pd.read_csv('data/True.csv')
fake['label'] = 'fake'
real['label'] = 'real'

df = pd.concat([fake, real]).sample(frac=1, random_state=42).reset_index(drop=True)
df['label_binary'] = (df['label'] == 'fake').astype(int)

# ISOT articles sometimes have a Reuters dateline prefix like
# 'WASHINGTON (Reuters) - ...' — strip it so it doesn't skew entity extraction
df['text'] = df['text'].str.replace(
    r'^[A-Z\s,]+\([^)]+\)\s*-\s*', '', regex=True
).str.strip()

print(f'Total articles : {len(df)}')
print(f'Class balance  : {df["label_binary"].value_counts().to_dict()}')
print(f'Columns        : {list(df.columns)}')
df[['title', 'text', 'label']].head(3)

Total articles : 44898
Class balance  : {1: 23481, 0: 21417}
Columns        : ['title', 'text', 'subject', 'date', 'label', 'label_binary']


,title,text,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",fake
1,Trump drops Steve Bannon from National Securit...,U.S. President Donald Trump removed his chief ...,real
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,real


## Step 2: Fact Extraction

In [3]:
def extract_facts(text: str) -> list[tuple[str, np.ndarray]]:
    """Split text into sentences and embed each one."""
    sentences = sent_tokenize(text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    if not sentences:
        return []
    embeddings = ENCODER.encode(sentences, batch_size=32, show_progress_bar=False)
    return list(zip(sentences, embeddings))

## Step 3: NLI-Typed Knowledge Graph

Edges typed as entailment / contradiction / neutral using a CrossEncoder NLI model. NLI only runs on pairs above the cosine similarity threshold — pairs below 0.60 are too topically distant to be logically related.

In [4]:
def build_knowledge_graph(facts: list[tuple[str, np.ndarray]],
                           sim_threshold: float = 0.60,
                           use_nli: bool = True) -> nx.Graph:
    G = nx.Graph()
    if not facts:
        return G

    sentences, embeddings = zip(*facts)
    embeddings = np.array(embeddings)

    for i, (sent, emb) in enumerate(zip(sentences, embeddings)):
        G.add_node(i, text=sent, embedding=emb, source='input', weight=1.0)

    sim_matrix = cosine_similarity(embeddings)
    candidate_pairs = [
        (i, j) for i in range(len(sentences))
        for j in range(i + 1, len(sentences))
        if sim_matrix[i, j] > sim_threshold
    ]

    if use_nli and candidate_pairs:
        pairs_text = [(sentences[i], sentences[j]) for i, j in candidate_pairs]
        all_logits = NLI_MODEL.predict(pairs_text)
        all_probs  = np.exp(all_logits) / np.exp(all_logits).sum(axis=1, keepdims=True)
        labels     = np.argmax(all_probs, axis=1)
        confs      = all_probs[np.arange(len(labels)), labels]
        for (i, j), label, conf in zip(candidate_pairs, labels, confs):
            etype = EDGE_CONTRADICTION if label == NLI_CONTRADICTION else \
                    EDGE_ENTAILMENT    if label == NLI_ENTAILMENT    else \
                    EDGE_NEUTRAL
            G.add_edge(i, j, similarity=float(sim_matrix[i, j]),
                       edge_type=etype, nli_confidence=float(conf))
    else:
        for i, j in candidate_pairs:
            G.add_edge(i, j, similarity=float(sim_matrix[i, j]),
                       edge_type=EDGE_NEUTRAL, nli_confidence=0.5)
    return G

## Step 4: Entity-Centric Claim Extraction

Queries are built from named entity combinations rather than dependency-parsed predicate structures. Search engines index by entity mention in titles and headers — `'Manchin Biden Build Back Better'` returns relevant results; `'Manchin block plan'` does not.

Sentences are scored by entity richness (PERSON=2.0, ORG/LAW/EVENT=1.5, GPE=1.0, DATE/CARDINAL=0.3). The top-scoring sentences have their entities extracted in priority order, deduplicated, and joined into a compact query string.

In [5]:
ENTITY_PRIORITY    = ['PERSON', 'ORG', 'LAW', 'EVENT', 'PRODUCT',
                       'GPE', 'FAC', 'NORP', 'CARDINAL', 'DATE']
MIN_VERIFIABLE     = {'PERSON', 'ORG', 'GPE', 'LAW', 'EVENT', 'PRODUCT', 'FAC', 'NORP'}
ENTITY_WEIGHTS     = {
    'PERSON': 2.0, 'ORG': 1.5, 'LAW': 1.5, 'EVENT': 1.5, 'PRODUCT': 1.5,
    'GPE': 1.0, 'FAC': 1.0, 'NORP': 1.0, 'CARDINAL': 0.3, 'DATE': 0.3
}


def score_sentence_for_search(sent) -> float:
    """Score a sentence by entity richness. Higher = better search query."""
    seen, score = set(), 0.0
    for ent in sent.ents:
        if ent.text not in seen and ent.label_ in ENTITY_WEIGHTS:
            score += ENTITY_WEIGHTS[ent.label_]
            seen.add(ent.text)
    if len(list(sent)) > 30:
        score -= 0.5  # penalize long rambling sentences
    return score


def build_entity_query(sent, max_tokens: int = 6) -> str | None:
    """
    Build a search query from named entities in priority order.
    Deduplicates overlapping entity spans (e.g. skips 'Biden' if 'Joe Biden' already added).
    Returns None if no verifiable entity types are present.
    """
    by_type: dict[str, list[str]] = {}
    for ent in sent.ents:
        if ent.label_ in ENTITY_PRIORITY:
            clean = ent.text.strip().rstrip('.,;:')
            if len(clean) > 1:
                by_type.setdefault(ent.label_, []).append(clean)

    if not by_type or not (set(by_type.keys()) & MIN_VERIFIABLE):
        return None

    selected, token_count = [], 0
    for etype in ENTITY_PRIORITY:
        for entity_text in by_type.get(etype, []):
            tokens = entity_text.split()
            if token_count + len(tokens) > max_tokens:
                continue
            already_covered = any(
                entity_text.lower() in ex.lower() or ex.lower() in entity_text.lower()
                for ex in selected
            )
            if not already_covered:
                selected.append(entity_text)
                token_count += len(tokens)
        if token_count >= max_tokens:
            break

    return ' '.join(selected) if selected else None


def extract_atomic_claims(text: str, max_claims: int = 3) -> list[str]:
    """
    Distill an article into compact, entity-centric search queries.
    Deduplicates queries that share >70% of their words.
    """
    doc = nlp(text[:5000])
    scored = [
        (score_sentence_for_search(s), s)
        for s in doc.sents
        if len(s.text.strip()) >= 20
    ]
    scored = [(sc, s) for sc, s in scored if sc > 0]
    scored.sort(key=lambda x: x[0], reverse=True)

    claims, seen_queries = [], set()
    for _, sent in scored[:max_claims * 3]:
        if len(claims) >= max_claims:
            break
        query = build_entity_query(sent)
        if query is None:
            # Fallback: capitalized noun chunks
            chunks = [
                c.text.strip() for c in sent.noun_chunks
                if any(t.text[0].isupper() for t in c if not t.is_stop)
                and len(c.text.strip()) > 3
            ]
            query = ' '.join(chunks[:3])[:80] if chunks else None
        if not query:
            continue
        query_words = set(query.lower().split())
        duplicate = any(
            len(query_words & set(ex.lower().split())) / max(len(query_words), 1) > 0.7
            for ex in seen_queries
        )
        if not duplicate:
            claims.append(query)
            seen_queries.add(query)
    return claims


# Sanity check on a real-looking sentence
sample = ('President Trump signed an executive order on immigration on Friday. '
          'The order was challenged by the ACLU in federal court. '
          'Senator Warren called it unconstitutional.')
print('Sample claims:', extract_atomic_claims(sample))

Sample claims: ['Trump Friday', 'Warren', 'ACLU']


## Step 5: Source Credibility Database

In [6]:
DB_PATH      = 'source_credibility.db'
MODEL_WEIGHT = 0.3
USER_WEIGHT  = 1.0


@contextmanager
def get_db():
    conn = sqlite3.connect(DB_PATH)
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()


def init_db():
    with get_db() as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS sources (
                domain        TEXT PRIMARY KEY,
                alpha         REAL DEFAULT 2.0,
                beta          REAL DEFAULT 2.0,
                model_updates INTEGER DEFAULT 0,
                user_updates  INTEGER DEFAULT 0,
                last_updated  TEXT
            )''')
        conn.execute('''
            CREATE TABLE IF NOT EXISTS credibility_log (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                domain      TEXT,
                signal_type TEXT,
                signal      TEXT,
                confidence  REAL,
                timestamp   TEXT
            )''')

init_db()


def extract_domain(url: str) -> str:
    return urlparse(url).netloc.replace('www.', '')


def get_credibility(domain: str) -> float:
    with get_db() as conn:
        row = conn.execute(
            'SELECT alpha, beta FROM sources WHERE domain = ?', (domain,)
        ).fetchone()
    return (row[0] / (row[0] + row[1])) if row else 0.5


def update_credibility(domain: str, signal: str, signal_type: str, confidence: float = 1.0):
    assert signal in ('real', 'fake') and signal_type in ('model', 'user')
    weight = (MODEL_WEIGHT if signal_type == 'model' else USER_WEIGHT) * confidence
    now    = datetime.utcnow().isoformat()
    with get_db() as conn:
        conn.execute(
            'INSERT OR IGNORE INTO sources (domain,alpha,beta,last_updated) VALUES(?,2.0,2.0,?)',
            (domain, now)
        )
        col        = 'alpha' if signal == 'real' else 'beta'
        update_col = 'model_updates' if signal_type == 'model' else 'user_updates'
        conn.execute(
            f'UPDATE sources SET {col}={col}+?,{update_col}={update_col}+1,last_updated=? WHERE domain=?',
            (weight, now, domain)
        )
        conn.execute(
            'INSERT INTO credibility_log(domain,signal_type,signal,confidence,timestamp) VALUES(?,?,?,?,?)',
            (domain, signal_type, signal, confidence, now)
        )


def bulk_update_from_prediction(scored_docs, prediction: float,
                                 model_confidence: float,
                                 confidence_threshold: float = 0.1) -> int:
    if model_confidence < confidence_threshold:
        return 0
    signal = 'fake' if prediction > 0.5 else 'real'
    n = 0
    for doc, doc_score in scored_docs:
        url = doc.get('link', '')
        if url:
            update_credibility(extract_domain(url), signal, 'model',
                               confidence=model_confidence * doc_score)
            n += 1
    return n


print('Credibility DB ready at', DB_PATH)

Credibility DB ready at source_credibility.db


## Step 6: DuckDuckGo Search with Disk Cache

Two fixes from models1:
1. Import is now `ddgs` (the package was renamed upstream from `duckduckgo_search`).
2. `os.makedirs(CACHE_DIR)` lives inside `ddg_search` rather than in a separate cell. This means the cache directory is always created before a write is attempted, even after a kernel restart where only the search cell is re-run.

In [ ]:
%pip intall --quiet ddgs

from duckduckgo_search import DDGS   # renamed from duckduckgo_search

CACHE_DIR = 'search_cache'


def _cache_path(query: str) -> str:
    return os.path.join(CACHE_DIR, hashlib.md5(query.encode()).hexdigest() + '.json')


def ddg_search(query: str, num_results: int = 10) -> list[dict]:
    """
    Search DuckDuckGo with transparent disk caching.
    Cache dir is created here — safe across kernel restarts.
    """
    os.makedirs(CACHE_DIR, exist_ok=True)  # inside the function, not a separate cell

    cache_file = _cache_path(query)
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            return json.load(f)

    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=num_results):
            results.append({
                'link':    r.get('href', ''),
                'title':   r.get('title', ''),
                'snippet': r.get('body', '')
            })

    with open(cache_file, 'w') as f:
        json.dump(results, f)

    time.sleep(0.5)  # gentle rate limiting; only applies on first fetch
    return results


def collect_evidence(text: str, k: int = 10) -> list[dict]:
    claims = extract_atomic_claims(text, max_claims=3)
    if not claims:
        return []
    seen_links, all_results = set(), []
    for claim in claims:
        try:
            results = ddg_search(claim, num_results=k)
        except Exception as e:
            print(f'  Search failed for "{claim[:60]}": {e}')
            continue
        for r in results:
            if r.get('link') and r['link'] not in seen_links:
                seen_links.add(r['link'])
                all_results.append(r)
    return all_results

## Step 7: DBSCAN Consensus Document Scoring

In [9]:
def score_documents_dbscan(documents: list[dict],
                            credibility_alpha: float = 0.6,
                            eps: float = 0.3,
                            min_samples: int = 2) -> list[tuple[dict, float]]:
    """
    Rank documents by DBSCAN consensus cluster membership blended with domain credibility.
    Consensus cluster members are scored by centrality; outliers get 0.1.
    """
    snippets = [doc.get('snippet', '') for doc in documents]
    if not snippets:
        return []

    embeddings  = ENCODER.encode(snippets, batch_size=32, show_progress_bar=False)
    sim_matrix  = cosine_similarity(embeddings)
    dist_matrix = np.clip(1.0 - sim_matrix, 0, 2).astype(np.float64)

    labels          = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed').fit_predict(dist_matrix)
    unique_labels   = [l for l in set(labels) if l != -1]
    consensus_label = max(unique_labels, key=lambda l: (labels == l).sum()) if unique_labels else None

    scored = []
    for i, doc in enumerate(documents):
        if consensus_label is not None:
            if labels[i] == consensus_label:
                members = np.where(labels == consensus_label)[0]
                cscore  = float(sim_matrix[i, members].mean())
            elif labels[i] == -1:
                cscore = 0.1
            else:
                cscore = float(sim_matrix[i].mean()) * 0.5
        else:
            cscore = float((sim_matrix[i].sum() - 1.0) / max(len(documents) - 1, 1))

        domain   = extract_domain(doc.get('link', ''))
        combined = credibility_alpha * cscore + (1 - credibility_alpha) * get_credibility(domain)
        scored.append((doc, float(combined)))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

## Step 8: Graph Augmentation with NLI-Typed Evidence Edges

In [10]:
def classify_edge_nli(sent_a: str, sent_b: str) -> tuple[int, float]:
    logits = NLI_MODEL.predict([(sent_a, sent_b)])[0]
    probs  = np.exp(logits) / np.exp(logits).sum()
    label  = int(np.argmax(probs))
    etype  = EDGE_CONTRADICTION if label == NLI_CONTRADICTION else \
             EDGE_ENTAILMENT    if label == NLI_ENTAILMENT    else EDGE_NEUTRAL
    return etype, float(probs[label])


def augment_knowledge_graph(G: nx.Graph,
                             scored_docs: list[tuple[dict, float]],
                             sim_threshold: float = 0.60,
                             score_threshold: float = 0.3,
                             use_nli: bool = True) -> nx.Graph:
    existing_nodes = list(G.nodes(data=True))
    next_id        = max(G.nodes()) + 1 if G.nodes() else 0
    new_nodes, new_edges = [], []

    for doc, doc_score in scored_docs:
        if doc_score < score_threshold or not doc.get('snippet'):
            continue
        for fact_text, fact_emb in extract_facts(doc['snippet']):
            nid = next_id
            next_id += 1
            new_nodes.append((nid, fact_text, fact_emb, doc_score))
            fact_emb_2d = fact_emb.reshape(1, -1)
            for eid, edata in existing_nodes:
                sim = float(cosine_similarity(fact_emb_2d, edata['embedding'].reshape(1, -1))[0, 0])
                if sim > sim_threshold:
                    etype, conf = classify_edge_nli(fact_text, edata['text']) if use_nli \
                                  else (EDGE_NEUTRAL, 0.5)
                    new_edges.append((nid, eid, sim, doc_score, etype, conf))

    for nid, text, emb, score in new_nodes:
        G.add_node(nid, text=text, embedding=emb, source='evidence', weight=score)
    for nid, eid, sim, score, etype, conf in new_edges:
        G.add_edge(nid, eid, similarity=sim, evidence_weight=score,
                   edge_type=etype, nli_confidence=conf)
    return G

## Step 9: Structural Node Features (14-dim) & PyG Conversion

In [11]:
import torch
from torch_geometric.data import Data


def compute_node_features(G: nx.Graph) -> np.ndarray:
    """
    14 structural features per node — no text content.
    0-7: basic evidence connectivity features
    8-13: NLI-typed entailment/contradiction breakdown
    """
    feature_list = []
    for node_id, data in G.nodes(data=True):
        neighbors     = list(G.neighbors(node_id))
        ev_nbrs       = [n for n in neighbors if G.nodes[n].get('source') == 'evidence']
        in_nbrs       = [n for n in neighbors if G.nodes[n].get('source') == 'input']
        ev_weights    = [G.edges[node_id, n].get('evidence_weight', 0.0) for n in ev_nbrs]
        all_sims      = [G.edges[node_id, n].get('similarity', 0.0) for n in neighbors]

        entailing     = [(n, G.edges[node_id, n]) for n in ev_nbrs
                         if G.edges[node_id, n].get('edge_type') == EDGE_ENTAILMENT]
        contradicting = [(n, G.edges[node_id, n]) for n in ev_nbrs
                         if G.edges[node_id, n].get('edge_type') == EDGE_CONTRADICTION]

        ent_wsum  = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5) for _, e in entailing)
        cont_wsum = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5) for _, e in contradicting)
        n_ev = max(len(ev_nbrs), 1)

        feature_list.append([
            float(len(ev_nbrs)),
            float(len(in_nbrs)),
            float(np.mean(ev_weights) if ev_weights else 0.0),
            float(np.max(ev_weights)  if ev_weights else 0.0),
            float(np.mean(all_sims)   if all_sims   else 0.0),
            float(len(ev_nbrs) / max(len(neighbors), 1)),
            1.0 if data.get('source') == 'evidence' else 0.0,
            float(data.get('weight', 1.0)),
            float(len(entailing)),
            float(len(contradicting)),
            ent_wsum,
            cont_wsum,
            float(len(entailing)    / n_ev),
            float(len(contradicting) / n_ev),
        ])

    arr     = np.array(feature_list, dtype=np.float32)
    col_max = arr.max(axis=0)
    col_max[col_max == 0] = 1
    return arr / col_max


def graph_to_pyg(G: nx.Graph, label: int | None = None) -> Data:
    node_ids = list(G.nodes())
    id_map   = {nid: i for i, nid in enumerate(node_ids)}
    x        = torch.tensor(compute_node_features(G), dtype=torch.float)

    if G.edges():
        edges      = [(id_map[u], id_map[v]) for u, v in G.edges()]
        edge_index = torch.tensor(edges, dtype=torch.long).T
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_feats = []
        for u, v in G.edges():
            ed  = G.edges[u, v]
            et  = ed.get('edge_type', EDGE_NEUTRAL)
            oh  = [1.0 if et == k else 0.0 for k in range(3)]
            edge_feats.append(oh + [
                ed.get('similarity', 0.0),
                ed.get('nli_confidence', 0.5),
                ed.get('evidence_weight', 0.0)
            ])
        ef_tensor = torch.tensor(edge_feats * 2, dtype=torch.float)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        ef_tensor  = torch.zeros((0, 6), dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=ef_tensor)
    if label is not None:
        data.y = torch.tensor([label], dtype=torch.float)
    return data

## Step 10: GAT Model — 4 Layers, Residual Connections, JumpingKnowledge

In [12]:
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool
from torch_geometric.loader import DataLoader


class FakeNewsGAT(torch.nn.Module):
    """
    4-layer GAT with residual connections, BatchNorm, JumpingKnowledge,
    and mean+max multi-pooling. Input: 14-dim structural features.
    """
    def __init__(self, input_dim=FEATURE_DIM, hidden_dim=64, n_heads=4, dropout=0.4):
        super().__init__()
        self.dropout    = dropout
        self.input_proj = torch.nn.Linear(input_dim, hidden_dim)

        self.conv1 = GATConv(hidden_dim,     hidden_dim,     heads=n_heads, concat=True,  dropout=dropout)
        self.lin1  = torch.nn.Linear(hidden_dim * n_heads, hidden_dim)
        self.bn1   = torch.nn.BatchNorm1d(hidden_dim)

        self.conv2 = GATConv(hidden_dim,     hidden_dim * 2, heads=n_heads, concat=True,  dropout=dropout)
        self.lin2  = torch.nn.Linear(hidden_dim * 2 * n_heads, hidden_dim * 2)
        self.bn2   = torch.nn.BatchNorm1d(hidden_dim * 2)

        self.conv3 = GATConv(hidden_dim * 2, hidden_dim * 2, heads=n_heads, concat=True,  dropout=dropout)
        self.lin3  = torch.nn.Linear(hidden_dim * 2 * n_heads, hidden_dim * 2)
        self.bn3   = torch.nn.BatchNorm1d(hidden_dim * 2)

        self.conv4 = GATConv(hidden_dim * 2, hidden_dim,     heads=n_heads, concat=False, dropout=dropout)
        self.bn4   = torch.nn.BatchNorm1d(hidden_dim)

        # JumpingKnowledge dim: 64+128+128+64=384; pool mean+max → 768
        pool_dim = (hidden_dim + hidden_dim*2 + hidden_dim*2 + hidden_dim) * 2

        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(pool_dim, 256),
            torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(256, 64),
            torch.nn.BatchNorm1d(64),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(64, 1)
        )

    def forward(self, x, edge_index, batch):
        x  = F.relu(self.input_proj(x))
        x  = F.dropout(x, p=self.dropout, training=self.training)

        x1 = self.bn1(F.relu(self.lin1(self.conv1(x,  edge_index)))) + x
        res2 = F.relu(torch.nn.functional.pad(x1, (0, x1.shape[1])))
        x2   = self.bn2(F.relu(self.lin2(self.conv2(x1, edge_index)))) + res2
        x3   = self.bn3(F.relu(self.lin3(self.conv3(x2, edge_index)))) + x2
        res4 = x3[:, :x1.shape[1]]
        x4   = self.bn4(F.relu(self.conv4(x3, edge_index))) + res4

        x_jk   = torch.cat([x1, x2, x3, x4], dim=1)
        x_pool = torch.cat([global_mean_pool(x_jk, batch),
                             global_max_pool(x_jk,  batch)], dim=1)
        return self.mlp(x_pool)


def train_gat(train_data, val_data, epochs=150, lr=5e-4, patience=20):
    model     = FakeNewsGAT()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.BCEWithLogitsLoss()
    t_loader  = DataLoader(train_data, batch_size=16, shuffle=True)
    v_loader  = DataLoader(val_data,   batch_size=16)

    best_val, no_improve, best_state = float('inf'), 0, None
    for epoch in range(epochs):
        model.train()
        t_loss = sum(
            (lambda out, loss: loss.item())(
                out := model(b.x, b.edge_index, b.batch).squeeze(),
                (lambda l: (l.backward(), torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0),
                             optimizer.step(), optimizer.zero_grad(), l)[4])
                (criterion(out, b.y.squeeze()))
            ) for b in t_loader
        ) if False else 0  # placeholder — use explicit loop below

        # Explicit training loop (cleaner than walrus-operator gymnastics above)
        model.train()
        t_loss = 0
        for b in t_loader:
            optimizer.zero_grad()
            out  = model(b.x, b.edge_index, b.batch).squeeze()
            loss = criterion(out, b.y.squeeze())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()
        scheduler.step()

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for b in v_loader:
                v_loss += criterion(
                    model(b.x, b.edge_index, b.batch).squeeze(), b.y.squeeze()
                ).item()

        avg_t, avg_v = t_loss / len(t_loader), v_loss / len(v_loader)
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}  train={avg_t:.4f}  val={avg_v:.4f}  '
                  f'lr={scheduler.get_last_lr()[0]:.2e}')

        if avg_v < best_val:
            best_val, no_improve = avg_v, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return model


def evaluate_model(model, dataset):
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for b in DataLoader(dataset, batch_size=16):
            probs = torch.sigmoid(model(b.x, b.edge_index, b.batch).squeeze())
            preds.extend(probs.tolist())
            labels.extend(b.y.squeeze().tolist())
    binary = [1 if p > 0.5 else 0 for p in preds]
    return {
        'accuracy': accuracy_score(labels, binary),
        'f1':       f1_score(labels, binary, zero_division=0),
        'auc':      roc_auc_score(labels, preds)
    }, preds


def save_model(model, path):
    torch.save(model.state_dict(), path)
    print(f'Saved to {path}')

def load_model(path):
    m = FakeNewsGAT()
    m.load_state_dict(torch.load(path, weights_only=True))
    m.eval()
    return m

## Step 11: Build Dataset

Using 500 articles (up from 200) — ISOT has 44k so we're no longer limited by dataset size. The cache means re-running this cell only costs time on the NLI/graph steps, not search.

In [13]:
from tqdm import tqdm


def build_dataset(df, text_col='text', label_col='label_binary',
                  sample=None, use_nli=True):
    if sample:
        # Stratified sample so class balance is preserved
        df = df.groupby(label_col, group_keys=False).apply(
            lambda g: g.sample(sample // 2, random_state=42)
        ).reset_index(drop=True)

    pyg_data, all_scored = [], []
    n_aug, n_fall = 0, 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Building graphs'):
        text, label = row[text_col], int(row[label_col])

        facts = extract_facts(text)
        if not facts:
            all_scored.append([])
            continue

        G = build_knowledge_graph(facts, use_nli=use_nli)

        scored_docs = []
        try:
            docs = collect_evidence(text, k=10)
            if docs:
                scored_docs = score_documents_dbscan(docs)
                G           = augment_knowledge_graph(G, scored_docs, use_nli=use_nli)
                n_aug += 1
            else:
                n_fall += 1
        except Exception as e:
            print(f'  Evidence failed: {e}')
            n_fall += 1

        pyg_data.append(graph_to_pyg(G, label=label))
        all_scored.append(scored_docs)

    print(f'Built {len(pyg_data)} graphs | Augmented: {n_aug} | Fallback: {n_fall}')
    return pyg_data, all_scored


pyg_dataset, all_scored_docs = build_dataset(df, sample=500)

C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\686160323.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(label_col, group_keys=False).apply(
Building graphs:   0%|          | 0/500 [00:00<?, ?it/s]C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarnin

  Search failed for "Empowered Group of Ministers (EGoM UID": Body collection error: error decoding response body


C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:   3%|▎         | 15/500 [00:59<55:39,  6.89s/it]C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been

  Search failed for "Council Venezuela Cuba Iran 47": https://www.bing.com/search ConnectError: ('error sending request for url (https://www.bing.com/search?q=Council+Venezuela+Cuba+Iran+47&first=41&FORM=PERE3)', 'https://www.bing.com/search?q=Council+Venezuela+Cuba+Iran+47&first=41&FORM=PERE3')


C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:  44%|████▍     | 222/500 [28:55<15:30:21, 200.80s/it]

...

  Search failed for "Nelson Mandela Thomas Sankara Macron Burkina": https://www.bing.com/search ConnectError: ('error sending request for url (https://www.bing.com/search?q=Nelson+Mandela+Thomas+Sankara+Macron+Burkina)', 'https://www.bing.com/search?q=Nelson+Mandela+Thomas+Sankara+Macron+Burkina')
  Search failed for "Macron s Francois Hollande La Francafrique": https://www.bing.com/search ConnectError: ('error sending request for url (https://www.bing.com/search?q=Macron+s+Francois+Hollande+La+Francafrique)', 'https://www.bing.com/search?q=Macron+s+Francois+Hollande+La+Francafrique')
  Search failed for "Macron Roch Marc Kabore Stones Burkina": https://www.bing.com/search ConnectError: ('error sending request for url (https://www.bing.com/search?q=Macron+Roch+Marc+Kabore+Stones+Burkina)', 'https://www.bing.com/search?q=Macron+Roch+Marc+Kabore+Stones+Burkina')
  Search failed for "Donald Trump Paul Ryan Obamacare House": https://www.bing.com/search ConnectError: ('error sending request

C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:  46%|████▌     | 230/500 [29:01<1:02:27, 13.88s/it]C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:  46%|████▌     | 231/500 [30:15<2:08:52, 28.75s/it]

  Search failed for "Donald Trump Vladimir Putin White House": https://www.bing.com/search TimeoutError: ('error sending request for url (https://www.bing.com/search?q=Donald+Trump+Vladimir+Putin+White+House)', 'https://www.bing.com/search?q=Donald+Trump+Vladimir+Putin+White+House')


C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:  46%|████▋     | 232/500 [30:27<1:47:59, 24.18s/it]C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\3372735843.py:23: RuntimeWarning: This package (`duckduckgo_search`) has b

Built 493 graphs | Augmented: 231 | Fallback: 262


## Step 12: Train, Evaluate, Update Credibility DB

In [14]:
indices             = list(range(len(pyg_dataset)))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_data = [pyg_dataset[i] for i in train_idx]
val_data   = [pyg_dataset[i] for i in val_idx]
test_data  = [pyg_dataset[i] for i in test_idx]
print(f'Split: {len(train_data)} train | {len(val_data)} val | {len(test_data)} test')

model = train_gat(train_data, val_data, epochs=150, patience=20)

metrics, test_probs = evaluate_model(model, test_data)
print(f'\nTest Results:')
print(f'  Accuracy : {metrics["accuracy"]:.3f}')
print(f'  F1       : {metrics["f1"]:.3f}')
print(f'  AUC-ROC  : {metrics["auc"]:.3f}')

# Update credibility DB from all splits
model.eval()
n_db = 0
with torch.no_grad():
    for batch, scored in zip(DataLoader(pyg_dataset, batch_size=1), all_scored_docs):
        if not scored:
            continue
        prob = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch).squeeze()).item()
        conf = abs(prob - 0.5) * 2
        n_db += bulk_update_from_prediction(scored, prob, conf, confidence_threshold=0.1)

print(f'\nCredibility DB: {n_db} source entries updated.')
save_model(model, 'fake_news_gat_isot.pt')

Split: 345 train | 74 val | 74 test


c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params

Epoch  10  train=0.7090  val=0.6893  lr=4.95e-04
Epoch  20  train=0.7044  val=0.7421  lr=4.78e-04
Early stopping at epoch 23

Test Results:
  Accuracy : 0.568
  F1       : 0.610
  AUC-ROC  : 0.531


C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\2198752891.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now    = datetime.utcnow().isoformat()
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\2198752891.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now    = datetime.utcnow().isoformat()
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\2198752891.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now    = datetime.utcnow().isoformat()
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\2198752891.py:


Credibility DB: 478 source entries updated.
Saved to fake_news_gat_isot.pt


C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\2198752891.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now    = datetime.utcnow().isoformat()
C:\Users\bhada\AppData\Local\Temp\ipykernel_35668\2198752891.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now    = datetime.utcnow().isoformat()


In [15]:
# Inspect credibility DB — should now have real domain names from ISOT evidence
with get_db() as conn:
    rows = conn.execute(
        'SELECT domain, alpha, beta, model_updates, user_updates '
        'FROM sources ORDER BY model_updates + user_updates DESC LIMIT 25'
    ).fetchall()

print(f'{"Domain":<40} {"Score":>6} {"Signals":>8} {"Model":>7} {"User":>6}')
print('-' * 70)
for domain, alpha, beta, m, u in rows:
    score = alpha / (alpha + beta)
    n     = int(alpha + beta - 4)
    print(f'{domain:<40} {score:>6.3f} {n:>8} {m:>7} {u:>6}')

Domain                                    Score  Signals   Model   User
----------------------------------------------------------------------
paul.fr                                   0.461        0      20      0
townhall.com                              0.475        0      20      0
en.wikipedia.org                          0.461        0      19      0
zhihu.com                                 0.479        0      13      0
britannica.com                            0.478        0      11      0
wowturkey.com                             0.472        0      10      0
lindsey.com                               0.480        0      10      0
tacomaworld.com                           0.488        0      10      0
biblegateway.com                          0.473        0      10      0
investinganswers.com                      0.485        0      10      0
emojipedia.org                            0.484        0      10      0
portal.ian.org.br                         0.473        0      10 